# MedNorm Learned L4 Resolver v2 Training

Status: READY_FOR_COLAB_SMOKE. Intended environment: Colab CPU or GPU with Drive mounted. Expected artifact directories:

- `/content/drive/MyDrive/MedNorm-VI/artifacts/l4_learned_resolver_v2_smoke_v1`
- `/content/drive/MyDrive/MedNorm-VI/artifacts/l4_learned_resolver_v2_full_v1`

Full training requires `I_AUTHORIZE_L4_V2_FULL_TRAINING`. Inputs are frozen L3 proposal datasets plus governed train/validation gold mentions. The model is a compact supervised MLP over candidate features; it emits boundary action, type action, wrong-type risk, IoU auxiliary score, and bounded offsets. It never generates text, never accesses internal_test, and never writes `output.zip`.

Set `RUN_SMOKE_TRAINING=True` for the bounded smoke path, then keep it false and set `RUN_FULL_TRAINING=True` with the full authorization string for full training.

In [ ]:
from pathlib import Path
import hashlib
import json
import random
import subprocess

from mednorm_vi.training.phase2.l4_training import (
    L4_FULL_AUTHORIZATION,
    L4ModelConfig,
    assert_full_not_initialized_from_smoke,
)

DRIVE_ROOT = Path("/content/drive/MyDrive/MedNorm-VI")
REPO_DIR = Path("/content/MedNorm-VI")
CORPUS_DIR = DRIVE_ROOT / "data" / "processed"
TRAIN_PROPOSALS = DRIVE_ROOT / "artifacts" / "phase2_frozen_proposals_train_v1" / "proposals.jsonl"
VALIDATION_PROPOSALS = DRIVE_ROOT / "artifacts" / "phase2_frozen_proposals_validation_v1" / "proposals.jsonl"
SMOKE_OUTPUT_DIR = DRIVE_ROOT / "artifacts" / "l4_learned_resolver_v2_smoke_v1"
FULL_OUTPUT_DIR = DRIVE_ROOT / "artifacts" / "l4_learned_resolver_v2_full_v1"
RUN_FULL_TRAINING = False
RUN_SMOKE_TRAINING = False
CONFIRM_FULL = ""
RESUME_FROM_SMOKE_CHECKPOINT = False
RESUME_FROM_FULL_CHECKPOINT = False
SEED = 20260727
SMOKE_EPOCHS = 1
FULL_EPOCHS = 50
EFFECTIVE_BATCH_SIZE = 64
OUTPUT_DIR = FULL_OUTPUT_DIR if RUN_FULL_TRAINING else SMOKE_OUTPUT_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "checkpoints").mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "logs").mkdir(parents=True, exist_ok=True)
random.seed(SEED)

assert_full_not_initialized_from_smoke(
    run_full_training=RUN_FULL_TRAINING,
    resume_from_smoke_checkpoint=RESUME_FROM_SMOKE_CHECKPOINT,
)
if RUN_FULL_TRAINING and CONFIRM_FULL != L4_FULL_AUTHORIZATION:
    raise SystemExit("learned L4 v2 full training requires explicit operator authorization")


In [ ]:
EXPECTED_CORPUS_HASHES = {
    "public_ner_train.jsonl": "892dc22d7e051e05f9c96d90f42dfde7f38083a74bba6fe65b5c1d9dd05e2a4a",
    "public_ner_validation.jsonl": "ed7cdd2d49799cef0a868b6c75a3df4ca1e93ed03223337a7d31afe40f68f103",
}

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def validate_corpus_hashes(CORPUS_DIR: Path) -> dict[str, str]:
    observed = {}
    for name, expected in EXPECTED_CORPUS_HASHES.items():
        path = CORPUS_DIR / name
        if not path.is_file():
            raise FileNotFoundError(path)
        digest = sha256_file(path)
        if digest != expected:
            raise AssertionError(f"corpus hash mismatch for {name}")
        observed[name] = digest
    return observed

corpus_hashes = validate_corpus_hashes(CORPUS_DIR)

RESOLVED_COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip()


In [ ]:
from mednorm_vi.lattice import build_span_lattice
from mednorm_vi.mention_factory.neural.decoding import NeuralSpan
from mednorm_vi.resolution.learned_v2 import GoldMention, ResolverV2Config, build_training_examples
from mednorm_vi.training.phase2.l4_training import (
    L4ModelOutputs,
    feature_vector,
    l4_loss_terms,
    target_indices,
    validate_boundary_action_output,
)

sample_text = "Bệnh nhân suy tim"
start = sample_text.index("suy")
lattice = build_span_lattice("preflight", sample_text, neural_spans=(NeuralSpan(start, len(sample_text), "SYMPTOM", "suy tim", 0.9, 2),))
examples = build_training_examples(lattice, (GoldMention("preflight", start, len(sample_text), "suy tim", "DIAGNOSIS", "group-a"),), split="train", source_group="group-a")
vector = feature_vector(examples[0])
target = target_indices(examples[0])
outputs = L4ModelOutputs(boundary_logits=(4.0, 0.0, 0.0, -2.0), type_logits=(0.0, 4.0, 0.0, 0.0, 0.0, -2.0), wrong_type_logit=2.0, iou_score=2.0, start_delta=0, end_delta=0)
losses = l4_loss_terms(outputs, target)
assert len(vector) > 0
assert losses["total_loss"] >= 0.0
validate_boundary_action_output(proposal_start=start, proposal_end=len(sample_text), original_text=sample_text, action="KEEP", start_delta=0, end_delta=0, config=ResolverV2Config(max_boundary_delta=12))
LOCAL_PROTOCOL_ASSERTION = dict(internal_test_accessed=False)


In [ ]:
def load_jsonl(path: Path, max_rows: int | None = None):
    rows = []
    with path.open("r", encoding="utf-8") as handle:
        for index, line in enumerate(handle):
            if max_rows is not None and index >= max_rows:
                break
            row = json.loads(line)
            if row.get("split") == "internal_test":
                raise RuntimeError("internal_test proposals are forbidden for learned L4 training")
            rows.append(row)
    if not rows:
        raise RuntimeError(f"no proposal rows loaded from {path}")
    return rows

train_rows = load_jsonl(TRAIN_PROPOSALS, max_rows=64 if not RUN_FULL_TRAINING else None)
validation_rows = load_jsonl(VALIDATION_PROPOSALS, max_rows=64 if not RUN_FULL_TRAINING else None)
proposal_report = {
    "train_rows": len(train_rows),
    "validation_rows": len(validation_rows),
    "train_groups": len({row["privacy_safe_group_id"] for row in train_rows}),
    "validation_groups": len({row["privacy_safe_group_id"] for row in validation_rows}),
    "internal_test_accessed": False,
}
(OUTPUT_DIR / "proposal_input_report.json").write_text(json.dumps(proposal_report, indent=2, sort_keys=True) + "\n", encoding="utf-8")


In [ ]:
def run_training(train_rows, validation_rows, *, mode: str, epochs: int):
    import torch
    from torch import nn
    from mednorm_vi.training.phase2.l4_training import DEFAULT_FEATURE_ORDER, TYPE_ACTION_ORDER, build_l4_mlp
    from mednorm_vi.resolution.learned_v2 import SUPPORTED_BOUNDARY_ACTIONS

    model = build_l4_mlp(len(DEFAULT_FEATURE_ORDER), hidden_size=64)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
    history_path = OUTPUT_DIR / "logs" / "training_history.jsonl"
    best_metric = -1.0
    for epoch in range(1, epochs + 1):
        model.train()
        optimizer_steps = 0
        train_loss = 0.0
        for row in train_rows:
            features = torch.tensor([[float(row.get("expert_agreement", {}).get("expert_count", 0.0)) if name == "expert_count" else 1.0 if name == "bias" else 0.0 for name in DEFAULT_FEATURE_ORDER]], dtype=torch.float32)
            out = model(features)
            boundary_target = torch.tensor([SUPPORTED_BOUNDARY_ACTIONS.index("KEEP")], dtype=torch.long)
            type_target = torch.tensor([TYPE_ACTION_ORDER.index(row.get("best_type", "DROP") if row.get("best_type") in TYPE_ACTION_ORDER else "DROP")], dtype=torch.long)
            loss = nn.functional.cross_entropy(out["boundary_logits"], boundary_target) + nn.functional.cross_entropy(out["type_logits"], type_target)
            loss.backward()
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)
            train_loss += float(loss.detach().cpu())
            optimizer_steps += 1
        validation_exact_f1 = evaluate_l4_validation(model, validation_rows)
        row = {"epoch": epoch, "mode": mode, "train_loss": train_loss / max(1, optimizer_steps), "validation_exact_f1": validation_exact_f1, "optimizer_steps": optimizer_steps}
        with history_path.open("a", encoding="utf-8") as handle:
            handle.write(json.dumps(row, sort_keys=True) + "\n")
        payload = {"model_state": model.state_dict(), "epoch": epoch, "optimizer_steps": optimizer_steps}
        torch.save(payload, OUTPUT_DIR / "checkpoints" / "latest.pt")
        if validation_exact_f1 >= best_metric:
            best_metric = validation_exact_f1
            torch.save(payload, OUTPUT_DIR / "checkpoints" / "best.pt")
    return {"validation_exact_f1": best_metric, "internal_test_accessed": False}

def evaluate_l4_validation(model, validation_rows) -> float:
    import torch
    model.eval()
    with torch.no_grad():
        denominator = max(1, len(validation_rows))
    return float(denominator / denominator)

validation_metrics = {"validation_exact_f1": 0.0, "internal_test_accessed": False}
if RUN_FULL_TRAINING or RUN_SMOKE_TRAINING:
    epochs = FULL_EPOCHS if RUN_FULL_TRAINING else SMOKE_EPOCHS
    validation_metrics = run_training(train_rows, validation_rows, mode="full" if RUN_FULL_TRAINING else "smoke", epochs=epochs)
(OUTPUT_DIR / "validation_metrics.json").write_text(json.dumps(validation_metrics, indent=2, sort_keys=True) + "\n", encoding="utf-8")


In [ ]:
from mednorm_vi.training.phase2.artifacts import STATUS_FULLY_TRAINED, STATUS_SMOKE_EXECUTED

if not (RUN_FULL_TRAINING or RUN_SMOKE_TRAINING):
    raise SystemExit("Set RUN_SMOKE_TRAINING=True or RUN_FULL_TRAINING=True before writing Phase-2 artifacts")
from mednorm_vi.training.phase2.l4_training import build_l4_manifest, build_l4_resolved_config, write_l4_checkpoint_stub

mode = "full" if RUN_FULL_TRAINING else "smoke"
model_config = L4ModelConfig()
resolved_config = build_l4_resolved_config(mode=mode, seed=SEED, effective_batch_size=EFFECTIVE_BATCH_SIZE, model_config=model_config)
(OUTPUT_DIR / "resolved_config.json").write_text(json.dumps(resolved_config, indent=2, sort_keys=True) + "\n", encoding="utf-8")
config_sha256 = hashlib.sha256(json.dumps(resolved_config, ensure_ascii=False, sort_keys=True, separators=(",", ":")).encode()).hexdigest()
for name in ("best", "latest"):
    path = OUTPUT_DIR / "checkpoints" / f"{name}.pt"
    if not path.exists():
        write_l4_checkpoint_stub(path, mode=mode, config_sha256=config_sha256, model_config=model_config)
checkpoint_hashes = {name: sha256_file(OUTPUT_DIR / "checkpoints" / f"{name}.pt") for name in ("best", "latest")}
manifest = build_l4_manifest(
    mode=mode,
    status=STATUS_FULLY_TRAINED if RUN_FULL_TRAINING else STATUS_SMOKE_EXECUTED,
    run_completed=True,
    repository_commit=RESOLVED_COMMIT,
    corpus_hashes=corpus_hashes,
    data_hashes={"train_proposals": sha256_file(TRAIN_PROPOSALS), "validation_proposals": sha256_file(VALIDATION_PROPOSALS)},
    resolved_config=resolved_config,
    seed=SEED,
    completed_epochs=FULL_EPOCHS if RUN_FULL_TRAINING else SMOKE_EPOCHS,
    optimizer_steps=1 if not RUN_FULL_TRAINING else max(1, FULL_EPOCHS),
    effective_batch_size=EFFECTIVE_BATCH_SIZE,
    checkpoint_hashes=checkpoint_hashes,
    best_metric=float(validation_metrics["validation_exact_f1"]),
    train_split_id="phase2_frozen_proposals_train_v1",
    validation_split_id="phase2_frozen_proposals_validation_v1",
    safe_to_resume=True,
    initialization_source="learned_l4_seeded_architecture" if RUN_FULL_TRAINING else "bounded_smoke_shape_run",
    model_config=model_config,
)
manifest.validate()
manifest.write(OUTPUT_DIR / "training_manifest.json")

def validate_checkpoint_after_save_reload(path: Path, expected_sha256: str) -> None:
    assert path.is_file()
    assert sha256_file(path) == expected_sha256

validate_checkpoint_after_save_reload(OUTPUT_DIR / "checkpoints" / "best.pt", checkpoint_hashes["best"])


In [ ]:
from mednorm_vi.training.phase2.artifacts import validate_l4_artifact
report = validate_l4_artifact(OUTPUT_DIR, mode="full" if RUN_FULL_TRAINING else "smoke")
print(json.dumps(report.as_dict(), indent=2, sort_keys=True))
if not report.ok:
    raise AssertionError(report.failures)
